In [14]:
import os
import random
import tempfile

import dagshub
import mlflow
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, r2_score , root_mean_squared_error
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "CPU")

print(f"Using device :- {device}")

Using device :- cuda


In [16]:
X_train = pd.read_csv("../data/processed/features.csv")
X_test  = pd.read_csv("../data/processed/features_test.csv")
y_train = pd.read_csv("../data/processed/labels.csv")
y_test = pd.read_csv("../data/processed/labels_test.csv")

# **Create DataSet & DataLoader Class**

In [17]:
class customDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features.values, dtype=torch.float32)
        self.labels = torch.tensor(labels.values, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [18]:
# Creating training dataset object
train_dataset = customDataset(X_train , y_train)
# Creating testing dataset object
test_dataset = customDataset(X_test , y_test)

In [19]:
print("len of training dataset :- ",len(train_dataset))
print("len of testing dataset :- ",len(test_dataset))

len of training dataset :-  1125732
len of testing dataset :-  281434


In [20]:
# num_workers= min(6 , os.cpu_count())

# Creating training dataloader
train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True,
    pin_memory=True,
    # num_workers=num_workers
)

# Creating testing DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=1024,
    shuffle=True,
    pin_memory=True,
    # num_workers=num_workers
)

# **Designing ANN Neural Network**

In [21]:
class ANN(nn.Module):
    def __init__(self , num_features):
        super().__init__()
        self.model = nn.Sequential(
            # Input Layer
            nn.Linear(num_features.shape[1] , 512),
            nn.BatchNorm1d(512),
            nn.GELU(),

            nn.Linear(512 , 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.1),

            nn.Linear(512 , 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.3),

            # Output Layer
            nn.Linear(256 , 1) 
        )
        
    def forward(self , x):
        return self.model(x)

In [22]:
# Setting lr and epochs
lr = 0.001
epochs = 50

# **Moving Model To GPU**

In [23]:
# instantiate the model
model = ANN(X_train)

# move model to GPU
model.to(device=device)

# Defining Loss Func
criterion = nn.MSELoss()

# Defining Optimizer 
optimizer = optim.Adam(model.parameters() , lr = lr)

In [24]:
model.parameters

<bound method Module.parameters of ANN(
  (model): Sequential(
    (0): Linear(in_features=28, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=512, out_features=512, bias=True)
    (4): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): GELU(approximate='none')
    (6): Dropout(p=0.1, inplace=False)
    (7): Linear(in_features=512, out_features=256, bias=True)
    (8): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (9): GELU(approximate='none')
    (10): Dropout(p=0.3, inplace=False)
    (11): Linear(in_features=256, out_features=1, bias=True)
  )
)>

In [25]:
from tqdm import tqdm

for epoch in range(epochs):
    model.train()
    total_epoch_loss = 0.0

    batch_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{epochs}",
        leave=False
    )

    for batch_features, batch_labels in batch_bar:
        # Move data to GPU cleanly
        batch_features = batch_features.to(device, non_blocking=True)
        
        # Send labels to GPU and calculate log-space target directly
        log_batch_labels = torch.log1p(batch_labels.to(device, non_blocking=True))

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(batch_features)

        # Compute loss
        loss = criterion(outputs, log_batch_labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        current_loss = loss.item()
        total_epoch_loss += current_loss

        # Update progress bar
        batch_bar.set_postfix(
            batch_loss=f"{current_loss:.4f}"
        )

    avg_loss = total_epoch_loss / len(train_loader)

    tqdm.write(
        f"Epoch {epoch + 1}/{epochs} | Average Log Loss: {avg_loss:.4f}"
    )

Epoch 1/50 | Average Log Loss: 0.2254


Epoch 2/50 | Average Log Loss: 0.1258


Epoch 3/50 | Average Log Loss: 0.1166


Epoch 4/50 | Average Log Loss: 0.1114


Epoch 5/50 | Average Log Loss: 0.1070


...

Epoch 46/50 | Average Log Loss: 0.0833


Epoch 47/50 | Average Log Loss: 0.0832


Epoch 48/50 | Average Log Loss: 0.0832


Epoch 49/50 | Average Log Loss: 0.0831


Epoch 50/50 | Average Log Loss: 0.0830


In [26]:
model.eval()
all_predictions = []
all_targets = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device)
        
        # Model outputs a log-space prediction
        log_predictions = model(batch_features).cpu().numpy()
         
        # Invert the prediction back to real-world trip minutes
        real_predictions = np.expm1(log_predictions)
        
        all_predictions.extend(real_predictions)
        all_targets.extend(batch_labels.numpy())

# Calculate your standard production metrics on the true scale
from sklearn.metrics import mean_absolute_error, r2_score

# all_targets -> y_test
# all_predictions -> y_pred
final_mae = mean_absolute_error(all_targets, all_predictions)
final_rmse = root_mean_squared_error(all_targets, all_predictions)
final_r2 = r2_score(all_targets, all_predictions)

print(f"Real-world MAE: {final_mae:.2f} mins")
print(f"Real-world RMSE: {final_rmse:.2f} mins")
print(f"Real-world R2 Score: {final_r2 * 100:.2f}%")

Real-world MAE: 3.18 mins
Real-world RMSE: 5.22 mins
Real-world R2 Score: 77.10%
